In [1]:
from astropy.io import fits
import numpy as np
from astropy.wcs import WCS 
from astropy import units as u
from astropy.wcs.utils import proj_plane_pixel_scales 
from astropy.stats import sigma_clip
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

In [2]:
ngc3034 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc3034_mips24_image_v5-0.fits'
ngc3031 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc3031_mips24_image_v5-0.fits'
ngc4254 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc4254_mips24_image_v5-0.fits'
ngc4594 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc4594_mips24_image_v5-0.fits'
ngc4826 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc5055_mips24_image_v5-0.fits'
ngc5055 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc5055_mips24_image_v5-0.fits'
ngc5195 = '/Users/valerie/Projet exp L3/Calcul_SFR/24 microns - MIPS/ngc5195_mips24_image_v5-0.fits'

In [3]:
hdu3034 = fits.open(ngc3034)
hdu3031 = fits.open(ngc3031)
hdu4254 = fits.open(ngc4254)
hdu4594 = fits.open(ngc4594)
hdu4826 = fits.open(ngc4826)
hdu5055 = fits.open(ngc5055)
hdu5195 = fits.open(ngc5195)

In [4]:
galaxies = np.array([hdu3034,hdu3031,hdu4254,hdu4594,hdu4826, hdu5055, hdu5195])
hdr = [galac[0].header for galac in galaxies]
print(hdulist[0].header.get("BUNIT")) #unite des pixels : intensité par unité d'angle solide, ici on obtient MJy/sr

In [5]:
wsc = [WCS(h) for h in hdr] #le header du FITS contient les infos du genre où pointe le téléscope, l'orientation, l'échelle... tout ça est contenu dans l'objet WCS

In [6]:
pixscale_deg = [proj_plane_pixel_scales(w)[0] for w in wsc] #taille d'un pixel en °, [0] car sinon il renverrait un array pour (échelle x, échelle y)
pixscale_arcsec = [p*3600 for p in pixscale_deg] #pixscale_arcsec est la taille angulaire d'un pixel en arcsec

In [7]:
pixscale_rad = [p/206265 for p in pixscale_arcsec] #conversion en radian
omega_pix = [p**2 for p in pixscale_rad] #angle solide = theta_x  * theta_y 

In [8]:
im_galaxies = [h[0] for h in galaxies]
im_data = [i.data for i in im_galaxies]

In [9]:
#après avoir entouré chaque galaxie avec un rectangle graphiquement, on entre les dimensions des rectangles

xming = [650, 1050, 600, 670,600, 450,990]
xmaxg = [1200,1550, 800, 960,670, 720, 1030]
yming = [900,800, 1000, 1350,1130, 1050,1510]
ymaxg = [1500,1600, 1200, 1390,1200,1270,1550]

In [10]:
#pour le fit

def gauss(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

In [11]:
mus = []
sigmas = []

for i in range(len(im_data)):

    #Flatten + nettoyage
    data = im_data[i].ravel()
    data = data[np.isfinite(data)]

    #Sigma clipping
    clipped = sigma_clip(data, sigma=3, maxiters=5)

    #garder uniquement les valeurs non masquées
    sky = clipped.data[~clipped.mask]

    #Histogramme
    n, bins = np.histogram(sky, bins=100, density=False)
    centers = 0.5 * (bins[:-1] + bins[1:])

    #Guess initial
    mu0 = np.median(sky)
    sigma0 = np.std(sky)
    A0 = n.max()

    p0 = [A0, mu0, sigma0]

    #Fit
    params, _ = curve_fit(gauss, centers, n, p0=p0)

    A_fit, mu_fit, sigma_fit = params

    mus.append(mu_fit)
    sigmas.append(sigma_fit)


In [12]:
parasite = mus
stdpara = sigmas

In [13]:
select = [np.nansum(im_data[i][yming[i]:ymaxg[i],xming[i]:xmaxg[i]])for i in range(0,7)] 

In [14]:
parasite_sur_select = [parasite[i]*(xmaxg[i]-xming[i])*(ymaxg[i]-yming[i]) for i in range(0,7)] #on "retire" le ciel de notre sélection
stdpara_sur_select = [stdpara[i]*(xmaxg[i]-xming[i])*(ymaxg[i]-yming[i]) for i in range(0,7)]

In [15]:
flux = [select[i] - parasite_sur_select[i] for i in range(0,7)]
stdflux = stdpara_sur_select

In [16]:
flux_jy = [flux[i] * 1e6 * omega_pix[i] for i in range(0,7)] #conversion en Jy
stdflux_jy = [stdflux[i] * 1e6 * omega_pix[i] for i in range(0,7)]

In [17]:
redshift= [0.00073, -0.00016, 0.008098,0.003659,0.001462,0.001669,0.00191]
D_Mpc = [cosmo.luminosity_distance(z) for z in redshift]
#pour NGC3031 et NGC4254, le méthode d'estimation par le redshift n'est pas très fiable, on se réfère aux données NED
D_Mpc[1] = 3.24*u.Mpc 
D_Mpc[2] = 14.4*u.Mpc
D_m = [d.to('m').value for d in D_Mpc]

In [18]:
c = 3e8
lam = 24e-6
nu = c / lam

In [19]:
Lnu = [4 * np.pi * D_m[i]**2 * flux_jy[i] * 1e-26 for i in range(7)] #unité W.Hz^-1
stdLnu = [4 * np.pi * D_m[i]**2 * stdflux_jy[i] * 1e-26 for i in range(7)]
nuLnu_W = [nu * lnu for lnu in Lnu] #unité : W
stdnuLnu_W = [nu * lnu for lnu in stdLnu] 
nuLnu = [x/(3.846e26) for x in nuLnu_W] #unité : L_sol
stdnuLnu = [x/(3.846e26) for x in stdnuLnu_W] 

[np.float64(3463611912.8888535), np.float64(207961583.87258777), np.float64(3367667789.9485044), np.float64(584353391.1364768), np.float64(333406557.2861224), np.float64(1121757540.1786559), np.float64(343017137.2218163)]


In [26]:
L_IR = [6856*nuLnux**0.71 for nuLnux in nuLnu] #Bavouzet. et al (2008)
stdL_IR = [6856*0.71*nuLnu[i]**(0.71-1)*stdnuLnu[i] for i in range(0,7)] #propagation des in
L_IR_erg = [lir*3.828e33 for lir in L_IR] #conversion en erg
stdL_IR_erg = [lir*3.828e33 for lir in stdL_IR]

In [27]:
SFR = [4.5e-44 * lir for lir in L_IR_erg] #Kennicutt (1998)
stdSFR = [4.5e-44 *stdL_IR_erg[i] + 2.43e-44 *L_IR_erg[i] for i in range(0,7)] #propagation des incertitudes

In [28]:
print(SFR)
print(stdSFR)

[np.float64(7.003594634539257), np.float64(0.950655717411445), np.float64(6.865292011002794), np.float64(1.9796795896814101), np.float64(1.329128250897179), np.float64(3.145455460495382), np.float64(1.3562179328098392)]
[np.float64(3.831745141824567), np.float64(0.6523074429756546), np.float64(3.886679891910306), np.float64(1.1815622123629832), np.float64(0.7239019804030951), np.float64(1.7671602302138598), np.float64(0.7357951124716527)]
